In [20]:
import pandas as pd
import numpy as np

In [21]:
df = pd.read_csv("../data/data.csv")

In [22]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (7043, 21)

Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [23]:
print("\nData types:")
print(df.dtypes)


Data types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


In [24]:
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [25]:
print("\nTarget distribution:")
print(df["Churn"].value_counts())


Target distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


## Cleaning the data

In [26]:
# Remove customer ID — it has no predictive meaning
df = df.drop(columns=["customerID"])

In [27]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [28]:
# Remove rows with missing values
df = df.dropna()

In [29]:
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum().sum())

Dataset shape: (7032, 20)

Missing values:
0


In [30]:
print("\nTarget distribution:")
print(df["Churn"].value_counts())


Target distribution:
Churn
No     5163
Yes    1869
Name: count, dtype: int64


In [31]:
X = df.drop(columns=["Churn"])
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [32]:
from sklearn.model_selection import train_test_split

numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    column for column in X.columns
    if column not in numeric_features
]

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier  
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline


def evaluate_models(models, X_train, X_test, y_train, y_test, preprocessor):
    results = []
    trained_models = {}

    for name, classifier in models.items():

        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("classifier", classifier)
            ]
        )

        # Train
        pipeline.fit(X_train, y_train)

        # Predictions
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        # Metrics
        metrics = {
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1": f1_score(y_test, y_pred),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        }

        results.append(metrics)
        trained_models[name] = pipeline

    results_df = pd.DataFrame(results)

    return results_df, trained_models

In [47]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"

    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42,
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3
    ),

    "Support Vector Machine": SVC(
        probability=True,
        random_state=42,
        class_weight="balanced"
    ),  

    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Naive Bayes": GaussianNB()

}

In [48]:
results, trained_models = evaluate_models(
    models,
    X_train,
    X_test,
    y_train,
    y_test,
    preprocessor
)

print("\nModel Comparison")
print("================")

print(
    results.sort_values(
        by="ROC-AUC",
        ascending=False
    ).to_string(index=False)
)

d:\project\aws-ml-churn-prediction\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Model Comparison
                 Model  Accuracy  Precision   Recall       F1  ROC-AUC
   Logistic Regression  0.725657   0.490132 0.796791 0.606925 0.835093
     Gradient Boosting  0.781095   0.603774 0.513369 0.554913 0.832873
               XGBoost  0.786780   0.615625 0.526738 0.567723 0.830491
         Random Forest  0.764748   0.547884 0.657754 0.597813 0.813883
Support Vector Machine  0.729211   0.494017 0.772727 0.602711 0.805600
           Naive Bayes  0.684435   0.449275 0.828877 0.582707 0.804040
   K-Nearest Neighbors  0.761194   0.546798 0.593583 0.569231 0.781696
